# 04 · Análisis: correlaciones y segmentación

Combina las features de movimiento (`derived_features.json`) con las puntuaciones psicométricas
(`psychometric_summary.json`) y produce:

- **`correlations.json`** — matriz de correlación Pearson movimiento × salud mental (objetivo C).
- **`clusters.json`** — segmentación KMeans + proyección PCA 2D por participante (objetivo D).

In [1]:
import sys, os
sys.path.append(os.path.abspath('../src'))
import etl_utils as eu
import pandas as pd, numpy as np, json

def load(name):
    return pd.DataFrame(json.load(open(os.path.join(eu.DATA_OUT_DIR, name), encoding='utf-8')))

derived = load('derived_features.json')
psy = load('psychometric_summary.json')
parts = load('participants.json')[['participant_id', 'bmi', 'sex', 'bmi_group']]
df = derived.merge(psy, on='participant_id', how='left').merge(parts, on='participant_id', how='left')
print('Participantes combinados:', len(df))

Participantes combinados: 58


## 1. Correlaciones movimiento × salud mental → `correlations.json`

In [2]:
movement_feats = ['mean_intensity', 'accel_variance', 'active_fraction', 'sedentary_fraction',
                  'activity_fragmentation', 'circadian_stability', 'mobility_radius_m', 'weekend_delta']
psych_feats = ['sdq_emotional', 'sdq_conduct', 'sdq_hyperactivity', 'sdq_peer', 'sdq_prosocial',
               'sdq_total_difficulties', 'snap_inattention', 'snap_hyperactivity_impulsivity', 'snap_odd', 'bmi']

corr = df[movement_feats + psych_feats].corr(method='pearson')
# Submatriz movimiento (filas) x psicométrico (columnas)
sub = corr.loc[movement_feats, psych_feats]
corr_payload = {
    'rows': movement_feats,
    'cols': psych_feats,
    'matrix': [[(None if pd.isna(v) else round(float(v), 3)) for v in sub.loc[r]] for r in movement_feats],
}
eu.write_json(corr_payload, 'correlations.json')
sub.round(2)

  escrito correlations.json  (0.9 KB)


,sdq_emotional,sdq_conduct,sdq_hyperactivity,sdq_peer,sdq_prosocial,sdq_total_difficulties,snap_inattention,snap_hyperactivity_impulsivity,snap_odd,bmi
mean_intensity,-0.02,0.10,-0.09,0.07,0.13,-0.01,-0.13,-0.10,-0.19,0.04
accel_variance,0.19,0.05,-0.07,-0.13,0.06,0.05,0.03,-0.07,-0.13,-0.11
active_fraction,-0.11,0.07,-0.06,0.19,-0.04,-0.03,-0.14,0.03,-0.03,0.19
sedentary_fraction,0.11,-0.07,0.06,-0.19,0.04,0.03,0.14,-0.03,0.03,-0.19
activity_fragmentation,0.23,0.01,0.17,-0.22,-0.05,0.15,0.22,0.18,0.10,-0.25
circadian_stability,-0.05,-0.02,0.07,0.16,0.04,0.04,-0.08,-0.01,-0.05,0.11
mobility_radius_m,0.24,0.19,0.30,-0.13,-0.13,0.27,0.39,0.22,0.06,-0.20
weekend_delta,0.51,-0.57,-0.44,-0.78,0.29,-0.21,-0.53,-0.83,-0.54,0.06


## 2. Segmentación (KMeans + PCA) → `clusters.json`

Agrupa perfiles psicoconductuales usando features de movimiento y de salud mental.
El número de clusters `k` se elige por *silhouette*.

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

clust_feats = ['mean_intensity', 'accel_variance', 'active_fraction', 'activity_fragmentation',
               'mobility_radius_m', 'sdq_total_difficulties', 'snap_inattention',
               'snap_hyperactivity_impulsivity']
cdf = df.dropna(subset=clust_feats).reset_index(drop=True)
X = StandardScaler().fit_transform(cdf[clust_feats])
print('Participantes con datos completos para clustering:', len(cdf))

best_k, best_s = 3, -1
for k in range(2, 6):
    lab = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
    s = silhouette_score(X, lab)
    print(f'k={k}  silhouette={s:.3f}')
    if s > best_s:
        best_k, best_s = k, s
print('-> k elegido:', best_k)

Participantes con datos completos para clustering: 47


C:\Users\marco\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


k=2  silhouette=0.238


C:\Users\marco\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


k=3  silhouette=0.211


C:\Users\marco\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


k=4  silhouette=0.221


C:\Users\marco\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


k=5  silhouette=0.238
-> k elegido: 2


In [4]:
km = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit(X)
cdf['cluster'] = km.labels_.astype(int)
pca = PCA(n_components=2, random_state=42)
xy = pca.fit_transform(X)
cdf['pca_x'], cdf['pca_y'] = xy[:, 0], xy[:, 1]

points = [{'participant_id': r.participant_id, 'cluster': int(r.cluster),
           'pca_x': float(r.pca_x), 'pca_y': float(r.pca_y)} for r in cdf.itertuples()]

# Perfil medio de cada cluster (para describir los grupos en la web)
profiles = []
for c in range(best_k):
    g = cdf[cdf.cluster == c]
    prof = {'cluster': c, 'n': int(len(g))}
    prof.update({f: float(g[f].mean()) for f in clust_feats})
    profiles.append(prof)

clusters_payload = {
    'k': int(best_k),
    'features': clust_feats,
    'silhouette': round(float(best_s), 3),
    'explained_variance': [round(float(v), 3) for v in pca.explained_variance_ratio_],
    'points': points,
    'profiles': profiles,
}
eu.write_json(clusters_payload, 'clusters.json')
pd.DataFrame(profiles).round(2)

C:\Users\marco\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


  escrito clusters.json  (3.9 KB)


,cluster,n,mean_intensity,accel_variance,active_fraction,activity_fragmentation,mobility_radius_m,sdq_total_difficulties,snap_inattention,snap_hyperactivity_impulsivity
0,0,18,0.64,0.17,0.93,0.02,40.33,10.04,1.69,1.51
1,1,29,0.53,0.40,0.76,0.04,126.47,10.52,1.88,1.60
